
# Feature Engineering com LM (Produtos Industriais)

Este notebook gera as tabelas `dim_customers.parquet`, `dim_products.parquet`, `fact_sales.parquet` e `products_features.parquet`.

**Features Criadas:**
| Categoria | Campos | Total |
| --------- | ------ | ----- |
| Produto | 12 features técnicas + TF-IDF | 12 |
| Cliente | industria, expected_problems, company_size, maintenance_model | 4 |
| **Total** | **Campos derivados + embeddings** | **16** |


In [28]:
import pandas as pd
import ast
import re
from sklearn.preprocessing import MultiLabelBinarizer

PRODUCTS_PATH = "../data/trusted/products_trusted.parquet"
CUSTOMERS_PATH = "../data/trusted/customers_trusted.parquet"
SALES_PATH = '../data/trusted/sales_trusted.parquet'
FEATURES_PATH = "../data/refined/"

products = pd.read_parquet(PRODUCTS_PATH)
customers = pd.read_parquet(CUSTOMERS_PATH)
sales = pd.read_parquet(SALES_PATH)

print(f"Dados carregados: {products.shape}, {customers.shape}, {sales.shape}")

Dados carregados: (10000, 17), (5000, 15), (120000, 15)


In [29]:
products.columns


Index(['product_id', 'product_name', 'product_category', 'product_subcategory',
       'manufacturer', 'model', 'bearing_type', 'material', 'load_capacity',
       'max_speed', 'temperature_limit', 'problem_type', 'unit_cost',
       'list_price', 'technical_description', 'technical_features',
       'lm_product_description'],
      dtype='object')

In [30]:
# ============================================================
# FUNÇÃO PARA EXTRAIR CAMPOS TÉCNICOS DA DESCRIÇÃO
# ============================================================

def extract_technical_specs(text):
    specs = {
        'max_speed': None,
        'load_capacity': None,
        'temperature_limit': None
    }
    
    if not isinstance(text, str):
        return specs
    
    text_normalized = text.replace('\n', ' ').replace('  ', ' ')
    
    def parse_industrial_number(num_str):
        # Remove separadores de milhar (vírgula ou ponto) que não sejam decimais
        # Se o número tem uma vírgula/ponto seguida de exatamente 3 dígitos, é milhar
        # Ex: 13,346 -> 13346 | 9.834 -> 9834
        
        # Primeiro, removemos qualquer separador que pareça ser de milhar
        # (vírgula ou ponto seguido de 3 dígitos no final ou antes de outro separador)
        temp_num = re.sub(r'([,\.])(?=[0-9]{3}(?:[^0-9]|$))', '', num_str)
        
        # Agora tratamos o que sobrou como decimal se houver vírgula/ponto
        temp_num = temp_num.replace(',', '.')
        
        try:
            return float(temp_num)
        except ValueError:
            return None

    # Extração de RPM
    rpm_patterns = [
        r'velocidade máxima[:\s]+([0-9,\.]+)\s*RPM',
        r'([0-9,\.]+)\s*RPM\s*(?:máxima|max)',
        r'RPM[:\s]+([0-9,\.]+)',
    ]
    for pattern in rpm_patterns:
        match = re.search(pattern, text_normalized, re.IGNORECASE)
        if match:
            val = parse_industrial_number(match.group(1))
            if val is not None:
                specs['max_speed'] = val
                break
    
    # Extração de Carga
    load_patterns = [
        r'Capacidade de carga[:\s]+([0-9,\.]+)\s*N',
        r'carga[:\s]+([0-9,\.]+)\s*N',
        r'([0-9,\.]+)\s*N\s*(?:carga|load)',
    ]
    for pattern in load_patterns:
        match = re.search(pattern, text_normalized, re.IGNORECASE)
        if match:
            val = parse_industrial_number(match.group(1))
            if val is not None:
                specs['load_capacity'] = val
                break
            
    # Extração de Temperatura
    temp_patterns = [
        r'limite de temperatura[:\s]+([0-9,\.]+)\s*°C',
        r'temperatura[:\s]+([0-9,\.]+)\s*°C',
        r'([0-9,\.]+)\s*°C',
    ]
    for pattern in temp_patterns:
        match = re.search(pattern, text_normalized, re.IGNORECASE)
        if match:
            val = parse_industrial_number(match.group(1))
            if val is not None:
                specs['temperature_limit'] = val
                break
            
    return specs

print("\n" + "="*60)
print("EXTRAINDO CAMPOS TÉCNICOS DA DESCRIÇÃO")
print("="*60)

cols_to_extract = ['max_speed', 'load_capacity', 'temperature_limit']
products = products.drop(columns=[c for c in cols_to_extract if c in products.columns])

extracted_specs = products['lm_product_description'].apply(extract_technical_specs)
specs_df = pd.DataFrame(list(extracted_specs))
products = pd.concat([products.reset_index(drop=True), specs_df], axis=1)
print(f"\nCampos extraídos com sucesso:")
for col in cols_to_extract:
    print(f"  - {col}: {products[col].notna().sum()} / {len(products)}")

def fill_missing_safe(df, col, value):
    count = df[col].isna().sum()
    if isinstance(count, pd.Series): count = count.iloc[0]
    if count > 0:
        df[col] = df[col].fillna(value)
        print(f"✓ {col}: {count} valores preenchidos com {value}")

fill_missing_safe(products, 'max_speed', 0.0)
fill_missing_safe(products, 'load_capacity', 0.0)
fill_missing_safe(products, 'temperature_limit', 250.0)

products['load_capacity'] = products['load_capacity'].astype(float)
products['temperature_limit'] = products['temperature_limit'].astype(float)
products['max_speed'] = products['max_speed'].astype(float)

print("\nTodos os valores faltantes foram preenchidos e tipos corrigidos")


EXTRAINDO CAMPOS TÉCNICOS DA DESCRIÇÃO

Campos extraídos com sucesso:
  - max_speed: 10000 / 10000
  - load_capacity: 10000 / 10000
  - temperature_limit: 10000 / 10000

Todos os valores faltantes foram preenchidos e tipos corrigidos


In [31]:
problem_keywords = {
    "Vibração": ["vibration", "stability", "balance", "oscilação", "vibr"],
    "Desgaste": ["wear", "durability", "long life", "desgaste", "erosão"],
    "Superaquecimento": ["overheating", "superaquec", "alta temperatura", "heat build-up"],
    "Corrosão": ["corrosion", "stainless", "humidity", "corros"],
    "Contaminação": ["sealed", "hygiene", "food", "contam"],
}

def infer_supported_problems(text):
    if not isinstance(text, str) or not text: return ["Uso Geral"]
    text_lower = text.lower()
    supported = [p for p, ks in problem_keywords.items() if any(k in text_lower for k in ks)]
    return supported if supported else ["Uso Geral"]

products['supported_problems'] = products['lm_product_description'].apply(infer_supported_problems)
print("Distribuição de problemas encontrados:")
print(products['supported_problems'].apply(len).value_counts())

Distribuição de problemas encontrados:
supported_problems
1    10000
Name: count, dtype: int64


In [32]:
mlb_customers = MultiLabelBinarizer()
customer_problem_features = mlb_customers.fit_transform(customers['expected_problems'])
customer_problem_df = pd.DataFrame(
    customer_problem_features,
    columns=[f"problem_{c.lower().replace(' ', '_').replace('ã', 'a').replace('ç', 'c').replace('õ', 'o')}" for c in mlb_customers.classes_]
)
customers_ml = pd.concat([customers.reset_index(drop=True), customer_problem_df], axis=1)

mlb = MultiLabelBinarizer()
problem_features = mlb.fit_transform(products['supported_problems'])
problem_features_df = pd.DataFrame(problem_features, columns=[f"problem_{c}" for c in mlb.classes_])
products_ml = pd.concat([products.reset_index(drop=True), problem_features_df], axis=1)

In [33]:
products_ml['full_description'] = (
    'Rolamento ' + products_ml['product_name'].astype(str) + ' - ' + 
    products_ml['bearing_type'].astype(str) + ' em ' + 
    products_ml['material'].astype(str) + '. ' +
    'Problema: ' + products_ml['supported_problems'].astype(str) + '. ' +
    'Descrição: ' + products_ml['lm_product_description'].astype(str)
).fillna('')

products_ml.to_parquet(f"{FEATURES_PATH}/products_features.parquet", index=False)

In [34]:
products_ml.to_parquet(f"{FEATURES_PATH}/dim_products.parquet", index=False)
customers_ml.to_parquet(f"{FEATURES_PATH}/dim_customers.parquet", index=False)
print("Features geradas e salvas com sucesso")

Features geradas e salvas com sucesso


In [35]:
sales['sale_date'] = pd.to_datetime(sales['sale_date'])
sales['last_updated'] = pd.to_datetime(sales['last_updated'])
sales.to_parquet('../data/refined/fact_sales.parquet', index=False)
print("fact_sales.parquet criado em ../data/refined/")

fact_sales.parquet criado em ../data/refined/
